#  Handwritten Prescription NLP Model
### Pipeline: Image/PDF → OCR → Medicine Extraction → Patient Guidance

**This notebook covers:**
1. Installing dependencies
2. Loading prescription images or PDFs
3. OCR to extract handwritten text (using EasyOCR + Tesseract)
4. NLP parsing to extract medicine name, dosage, frequency
5. Generating patient-friendly medicine usage guidance using a language model
6. Exporting results to a structured report


---
##  Step 1: Install Dependencies

In [ ]:
# Install all required packages
!pip install easyocr pillow pdf2image pytesseract transformers torch pandas anthropic opencv-python-headless -q

# For PDF rendering (poppler)
import sys
if sys.platform == 'linux':
    !apt-get install -y poppler-utils tesseract-ocr -q
elif sys.platform == 'darwin':
    !brew install poppler tesseract

print(' All packages installed!')

---
##  Step 2: Import Libraries & Configuration

In [ ]:
import os
import re
import json
import base64
import warnings
import pandas as pd
import numpy as np
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from IPython.display import display, HTML, Markdown

warnings.filterwarnings('ignore')

# ── CONFIG ─────────────────────────────────────────────────
# Option A: Use Anthropic Claude API (recommended for best accuracy)
ANTHROPIC_API_KEY = "YOUR_API_KEY_HERE"   # <-- paste your key

# Option B: Use local OCR (EasyOCR/Tesseract) — set USE_CLAUDE = False
USE_CLAUDE_VISION = True    # True = Claude API  |  False = local OCR

# Path to your prescription images or PDF folder
PRESCRIPTION_FOLDER = "./prescriptions"   # <-- change this to your folder
OUTPUT_CSV = "./prescription_results.csv"

os.makedirs(PRESCRIPTION_FOLDER, exist_ok=True)
print(' Configuration ready!')
print(f'   Mode: {"Claude Vision API" if USE_CLAUDE_VISION else "Local OCR (EasyOCR + Tesseract)"}')
print(f'   Prescriptions folder: {PRESCRIPTION_FOLDER}')

---
##  Step 3: Load Prescription Images / PDFs

In [ ]:
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt
from pdf2image import convert_from_path

#  Your prescription folder
PRESCRIPTION_FOLDER = r"C:\Users\abidm\Downloads\Prescription"

#  Correct Poppler path (from your screenshot)
POPPLER_PATH = r"C:\poppler\poppler-25.12.0\Library\bin"


def load_prescription_files(folder_path):
    folder = Path(folder_path)
    prescriptions = []
    supported_images = [".jpg", ".jpeg", ".png", ".webp", ".bmp", ".tiff"]

    for file in sorted(folder.iterdir()):
        ext = file.suffix.lower()

        #  Load images
        if ext in supported_images:
            img = Image.open(file).convert("RGB")
            prescriptions.append((file.name, img))
            print(f" Loaded image: {file.name}")

        #  Load PDFs (FIXED)
        elif ext == ".pdf":
            try:
                pages = convert_from_path(
                    str(file),
                    dpi=300,
                    poppler_path=POPPLER_PATH
                )
                for i, page in enumerate(pages):
                    name = f"{file.stem}_page{i+1}.pdf"
                    prescriptions.append((name, page))
                    print(f" Loaded PDF page: {name}")
            except Exception as e:
                print(f"❌ Error loading {file.name}: {e}")

    print(f"\n Total prescriptions loaded: {len(prescriptions)}")
    return prescriptions


def preview_prescriptions(prescriptions, max_show=3):
    n = min(len(prescriptions), max_show)

    if n == 0:
        print(" No files found.")
        return

    fig, axes = plt.subplots(1, n, figsize=(6 * n, 6))

    if n == 1:
        axes = [axes]

    for i, (name, img) in enumerate(prescriptions[:n]):
        axes[i].imshow(img)
        axes[i].set_title(name)
        axes[i].axis("off")

    plt.suptitle("Prescription Preview", fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()


#  RUN
prescriptions = load_prescription_files(PRESCRIPTION_FOLDER)
preview_prescriptions(prescriptions)

---
##  Step 4: OCR — Extract Text from Handwriting

We support two OCR modes:
- **Claude Vision API** (recommended) — best accuracy for messy handwriting
- **Local EasyOCR + Tesseract** — works offline, no API key needed

In [ ]:
import io
def ocr_with_claude(pil_image, api_key):
    """Use Claude's vision to extract raw text from prescription image."""
    import anthropic

    # Convert PIL image to base64
    buffer = io.BytesIO()
    pil_image.save(buffer, format="JPEG", quality=95)
    img_b64 = base64.b64encode(buffer.getvalue()).decode()

    client = anthropic.Anthropic(api_key=api_key)
    response = client.messages.create(
        model="claude-opus-4-5",
        max_tokens=1000,
        messages=[{
            "role": "user",
            "content": [
                {"type": "image", "source": {"type": "base64", "media_type": "image/jpeg", "data": img_b64}},
                {"type": "text", "text": (
                    "This is a handwritten doctor's prescription. "
                    "Please carefully read and transcribe ALL text visible, "
                    "including medicine names, dosages, frequency, duration, and any instructions. "
                    "Preserve the structure as much as possible."
                )}
            ]
        }]
    )
    return response.content[0].text


def ocr_with_local(pil_image):
    """Use EasyOCR for handwriting and Tesseract as fallback."""
    import easyocr
    import pytesseract
    import cv2

    # Preprocess: grayscale + denoise + threshold
    img_array = np.array(pil_image)
    gray = cv2.cvtColor(img_array, cv2.COLOR_RGB2GRAY)
    denoised = cv2.fastNlMeansDenoising(gray, h=10)
    _, thresh = cv2.threshold(denoised, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

    # EasyOCR (better for handwriting)
    reader = easyocr.Reader(['en'], gpu=False)
    results = reader.readtext(thresh, detail=0, paragraph=True)
    easy_text = '\n'.join(results)

    # Tesseract fallback for printed text
    tess_text = pytesseract.image_to_string(
        Image.fromarray(thresh),
        config='--psm 6 --oem 3'
    )

    # Combine and deduplicate
    combined = easy_text if len(easy_text) > len(tess_text) else tess_text
    return combined.strip()


def extract_text(pil_image, filename):
    """Extract text from a prescription image."""
    print(f"  🔍 OCR on: {filename}", end=" ... ")
    try:
        if USE_CLAUDE_VISION and ANTHROPIC_API_KEY != "YOUR_API_KEY_HERE":
            text = ocr_with_claude(pil_image, ANTHROPIC_API_KEY)
        else:
            text = ocr_with_local(pil_image)
        print(f"✅ ({len(text)} chars)")
        return text
    except Exception as e:
        print(f"❌ Error: {e}")
        return ""


print("OCR functions ready! Run Step 5 to process prescriptions.")

---
##  Step 5: NLP — Extract Medicine Details

In [ ]:
import anthropic

def extract_medicines_from_text(raw_text, api_key):
    """
    Use Claude NLP to parse medicine names, dosage, frequency, duration
    from raw OCR text. Returns structured JSON.
    """
    client = anthropic.Anthropic(api_key=api_key)

    system_prompt = """You are a medical NLP system that extracts structured medicine data from prescription text.
Return ONLY valid JSON, no markdown, no explanation.
Format:
{
  "patient_name": "...",
  "doctor_name": "...",
  "date": "...",
  "diagnosis": "...",
  "medicines": [
    {
      "name": "Medicine name",
      "generic_name": "Generic/chemical name if identifiable",
      "dosage": "e.g. 500mg",
      "frequency": "e.g. Twice daily (BD)",
      "duration": "e.g. 5 days",
      "route": "e.g. Oral / Topical / IV",
      "timing": "e.g. After meals / Before bed",
      "special_instructions": "e.g. Do not crush tablet"
    }
  ],
  "additional_notes": "Diet advice, follow-up, restrictions etc."
}
If a field is unclear or missing, set it to null. Extract as much as possible."""

    response = client.messages.create(
        model="claude-opus-4-5",
        max_tokens=1500,
        system=system_prompt,
        messages=[{
            "role": "user",
            "content": f"Extract medicines from this prescription text:\n\n{raw_text}"
        }]
    )

    raw = response.content[0].text.strip()
    raw = raw.replace("```json", "").replace("```", "").strip()
    return json.loads(raw)


def extract_medicines_local(raw_text):
    """
    Fallback: rule-based NLP using regex patterns.
    Less accurate but works offline.
    """
    medicines = []
    dosage_pattern = r'(\d+\s*(?:mg|mcg|ml|g|IU|units?|tab(?:lets?)?|cap(?:sules?)?))'  
    freq_pattern = r'((?:once|twice|thrice|\d+\s*times?)[\s\w]*(?:daily|a\s*day|per\s*day|BD|TDS|OD|QID))'
    duration_pattern = r'((?:for\s*)?\d+\s*(?:days?|weeks?|months?))'

    lines = [l.strip() for l in raw_text.split('\n') if l.strip()]
    for line in lines:
        dosage = re.findall(dosage_pattern, line, re.IGNORECASE)
        freq   = re.findall(freq_pattern,   line, re.IGNORECASE)
        dur    = re.findall(duration_pattern, line, re.IGNORECASE)
        if dosage:
            name_match = re.match(r'^([A-Za-z][\w\s\-]+?)(?:\s+\d|\s*$)', line)
            medicines.append({
                "name": name_match.group(1).strip() if name_match else line[:30],
                "generic_name": None,
                "dosage": dosage[0] if dosage else None,
                "frequency": freq[0] if freq else None,
                "duration": dur[0] if dur else None,
                "route": "Oral",
                "timing": None,
                "special_instructions": None
            })

    return {
        "patient_name": None, "doctor_name": None, "date": None,
        "diagnosis": None, "medicines": medicines, "additional_notes": None
    }


print(" Medicine extraction functions ready!")

---
##  Step 6: Generate Patient Guidance

In [ ]:
def generate_patient_guidance(structured_data, api_key):
    """
    Generate clear, patient-friendly usage instructions for each medicine.
    Includes: how to take it, food interactions, common side effects, warnings.
    """
    client = anthropic.Anthropic(api_key=api_key)

    medicines_summary = json.dumps(structured_data.get("medicines", []), indent=2)
    diagnosis = structured_data.get("diagnosis", "Not specified")

    prompt = f"""Based on this prescription data:

Diagnosis: {diagnosis}

Medicines:
{medicines_summary}

Generate clear, patient-friendly guidance for each medicine. For each medicine include:
1. What it is used for
2. How and when to take it (timing with meals, water intake)
3. Common side effects to watch for
4. Important warnings or precautions
5. Food or drug interactions to avoid
6. What to do if a dose is missed

Use simple language a patient can understand. Format as a clear numbered list per medicine.
End with general safety reminders."""

    response = client.messages.create(
        model="claude-opus-4-5",
        max_tokens=2000,
        messages=[{"role": "user", "content": prompt}]
    )
    return response.content[0].text


def generate_guidance_local(structured_data):
    """Simple rule-based guidance fallback (no API)."""
    lines = ["##  Medicine Guidance\n"]
    for i, med in enumerate(structured_data.get("medicines", []), 1):
        lines.append(f"### {i}. {med.get('name', 'Unknown Medicine')}")
        if med.get('dosage'):    lines.append(f"- **Dose:** {med['dosage']}")
        if med.get('frequency'): lines.append(f"- **Frequency:** {med['frequency']}")
        if med.get('duration'):  lines.append(f"- **Duration:** {med['duration']}")
        if med.get('timing'):    lines.append(f"- **When to take:** {med['timing']}")
        if med.get('special_instructions'): lines.append(f"- **Note:** {med['special_instructions']}")
        lines.append("-  Consult your doctor before stopping any medicine.\n")
    lines.append("\n> Always follow your doctor's instructions. This is for reference only.")
    return '\n'.join(lines)


print(" Guidance generation functions ready!")

---
##  Step 7: Run Full Pipeline on All Prescriptions

In [ ]:
USE_API = (USE_CLAUDE_VISION and ANTHROPIC_API_KEY != "YOUR_API_KEY_HERE")

all_results = []  # store results for export

for idx, (filename, img) in enumerate(prescriptions):
    print(f"\n{'='*60}")
    print(f" Processing [{idx+1}/{len(prescriptions)}]: {filename}")
    print('='*60)

    # --- OCR ---
    if USE_API:
        raw_text = ocr_with_claude(img, ANTHROPIC_API_KEY)
    else:
        raw_text = ocr_with_local(img)

    print("\n Extracted Text:")
    print(raw_text[:600] + ("..." if len(raw_text) > 600 else ""))

    # --- NLP Extraction ---
    print("\n Extracting medicines...")
    try:
        if USE_API:
            structured = extract_medicines_from_text(raw_text, ANTHROPIC_API_KEY)
        else:
            structured = extract_medicines_local(raw_text)
        print(f"    Found {len(structured.get('medicines', []))} medicine(s)")
    except Exception as e:
        print(f"    Extraction error: {e}")
        structured = {"medicines": [], "additional_notes": None}

    # --- Guidance ---
    print("\n Generating patient guidance...")
    try:
        if USE_API:
            guidance = generate_patient_guidance(structured, ANTHROPIC_API_KEY)
        else:
            guidance = generate_guidance_local(structured)
    except Exception as e:
        guidance = f"Could not generate guidance: {e}"

    # --- Display ---
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    axes[0].imshow(img)
    axes[0].set_title(f'Prescription: {filename}', fontsize=11)
    axes[0].axis('off')

    meds = structured.get('medicines', [])
    table_data = [[m.get('name',''), m.get('dosage',''), m.get('frequency',''), m.get('duration','')] for m in meds]
    if table_data:
        axes[1].axis('off')
        tbl = axes[1].table(
            cellText=table_data,
            colLabels=['Medicine', 'Dosage', 'Frequency', 'Duration'],
            cellLoc='center', loc='center'
        )
        tbl.auto_set_font_size(False)
        tbl.set_fontsize(10)
        tbl.scale(1.2, 1.8)
        axes[1].set_title('Extracted Medicines', fontsize=11)
    else:
        axes[1].text(0.5, 0.5, 'No medicines detected', ha='center', va='center', fontsize=12)
        axes[1].axis('off')

    plt.tight_layout()
    plt.show()

    display(Markdown(f"###  Patient Guidance\n{guidance}"))

    # Save result
    for med in meds:
        all_results.append({
            'file': filename,
            'patient': structured.get('patient_name'),
            'doctor': structured.get('doctor_name'),
            'date': structured.get('date'),
            'diagnosis': structured.get('diagnosis'),
            **{k: med.get(k) for k in ['name','generic_name','dosage','frequency','duration','route','timing','special_instructions']},
            'guidance': guidance
        })

print(f"\n All {len(prescriptions)} prescription(s) processed!")

In [ ]:
import pytesseract

#  FIX PATH (this is the key line)
pytesseract.pytesseract.tesseract_cmd = r"C:\Program Files\Tesseract-OCR\tesseract.exe"


def extract_text_from_prescriptions(prescriptions):
    results = []

    for name, img in prescriptions:
        try:
            text = pytesseract.image_to_string(img)
            print(f"\n Extracted from {name}:\n{text[:200]}")
            results.append((name, text))
        except Exception as e:
            print(f" OCR failed for {name}: {e}")

    return results


#  RUN OCR
ocr_results = extract_text_from_prescriptions(prescriptions)

---
##  Step 8: View Summary & Export Results

In [ ]:
if all_results:
    df = pd.DataFrame(all_results)
    print(f" Total medicines extracted: {len(df)}")
    print(f" Unique prescriptions: {df['file'].nunique()}")
    print()
    display(df[['file','name','dosage','frequency','duration','timing']].fillna('-'))

    # Export to CSV
    df.to_csv(OUTPUT_CSV, index=False)
    print(f"\n Results saved to: {OUTPUT_CSV}")

    # Bar chart: medicines per prescription
    counts = df.groupby('file')['name'].count()
    counts.plot(kind='bar', figsize=(10, 4), color='#00C896', edgecolor='black')
    plt.title('Medicines per Prescription', fontsize=13)
    plt.ylabel('Number of Medicines')
    plt.xlabel('Prescription File')
    plt.xticks(rotation=30, ha='right')
    plt.tight_layout()
    plt.show()
else:
    print(" No results to display. Make sure prescriptions folder contains image/PDF files.")